In [ ]:
# use first lat, last lat, I should really have flights use the last lat, lon in the ADS-B sequence where it thinks there was a flight. 
# Also first lat, last lat has to be on American FAA airspace. 

In [5]:
from datetime import date

from data_engineering.flights.flight_type import get_flights
from flights.evaluation.american_evals import continental_usa_to_continental_usa
import polars as pl

target_date = date(2026,3,1)
df_flights = get_flights(target_date)
df_flights = df_flights.filter(pl.col("ladd") | pl.col("pia"))

all_flight_counts_by_icao = df_flights.group_by("icao").agg(pl.len().alias("all_flight_count"))
continental_flight_counts_by_icao = (
    continental_usa_to_continental_usa(df_flights)
    .group_by("icao")
    .agg(pl.len().alias("continental_flight_count"))
)
continental_only_icaos = (
    all_flight_counts_by_icao
    .join(continental_flight_counts_by_icao, on="icao", how="left")
    .filter(pl.col("all_flight_count") == pl.col("continental_flight_count"))
    .select("icao")
)

df_flights = df_flights.join(continental_only_icaos, on="icao", how="inner")

In [6]:
TELEPORT_MIN_DISTANCE_KM = 50.0
EARTH_RADIUS_KM = 6371.0088

prev_lat = pl.col("prev_last_lat").radians()
prev_lon = pl.col("prev_last_lon").radians()
curr_lat = pl.col("first_lat").radians()
curr_lon = pl.col("first_lon").radians()
haversine_a = (
    ((curr_lat - prev_lat) / 2).sin().pow(2)
    + prev_lat.cos() * curr_lat.cos() * ((curr_lon - prev_lon) / 2).sin().pow(2)
)

teleport_flights = (
    df_flights
    .sort(["icao", "takeoff_time", "landing_time"])
    .with_columns(
        pl.col("landing_time").shift(1).over("icao").alias("prev_landing_time"),
        pl.col("landing_airport_ident").shift(1).over("icao").alias("prev_landing_airport_ident"),
        pl.col("last_lat").shift(1).over("icao").alias("prev_last_lat"),
        pl.col("last_lon").shift(1).over("icao").alias("prev_last_lon"),
    )
    .filter(
        pl.all_horizontal(
            pl.col("prev_last_lat").is_not_null(),
            pl.col("prev_last_lon").is_not_null(),
            pl.col("first_lat").is_not_null(),
            pl.col("first_lon").is_not_null(),
        )
    )
    .with_columns(
        (2 * EARTH_RADIUS_KM * haversine_a.sqrt().clip(0, 1).arcsin()).alias("endpoint_jump_km"),
        ((pl.col("takeoff_time") - pl.col("prev_landing_time")).dt.total_seconds() / 3600).alias("time_gap_hours"),
    )
    .with_columns(
        (pl.col("endpoint_jump_km") / pl.col("time_gap_hours")).alias("required_ground_speed_kmh")
    )
    .filter(pl.col("endpoint_jump_km") >= TELEPORT_MIN_DISTANCE_KM)
    .select(
        "icao",
        "prev_landing_time",
        "takeoff_time",
        "time_gap_hours",
        "endpoint_jump_km",
        "required_ground_speed_kmh",
        "prev_landing_airport_ident",
        "takeoff_airport_ident",
        "prev_last_lat",
        "prev_last_lon",
        "first_lat",
        "first_lon",
        "pia",
        "ladd",
        "aircraft_type",
        "owner",
    )
    .sort("endpoint_jump_km", descending=True)
)

teleport_icaos = (
    teleport_flights
    .group_by("icao")
    .agg(
        pl.len().alias("teleport_count"),
        pl.max("endpoint_jump_km").round(1).alias("max_endpoint_jump_km"),
        pl.min("time_gap_hours").round(2).alias("min_time_gap_hours"),
        pl.col("takeoff_time").min().alias("first_teleport_takeoff_time"),
    )
    .sort("max_endpoint_jump_km", descending=True)
)

teleport_icaos

icao,teleport_count,max_endpoint_jump_km,min_time_gap_hours,first_teleport_takeoff_time
str,u32,f64,f64,datetime[ms]
"""a921a9""",1,1251.8,4.28,2026-03-01 23:56:26.210
"""a1f74c""",1,716.0,9.37,2026-03-01 13:43:58.645
"""a021e9""",1,151.9,2.91,2026-03-01 04:34:58.170
"""aa4084""",1,143.1,2.64,2026-03-01 17:35:00.380
"""a1f526""",1,126.1,14.09,2026-03-01 16:43:49.810
…,…,…,…,…
"""a6b59e""",1,53.8,1.59,2026-03-01 18:41:42.355
"""a96f2b""",1,52.0,1.58,2026-03-01 19:30:26.010
"""a3c231""",1,51.2,0.9,2026-03-01 19:34:30.550


In [8]:
df_flights.filter(pl.col("icao") == "a5824e")

icao,callsign,registration,takeoff_time,takeoff_airport_ident,takeoff_airport_ident_distance_km,landing_time,landing_airport_ident,landing_airport_ident_distance_km,first_lat,first_lon,first_baro_altitude_ft,first_geom_altitude_ft,last_lat,last_lon,last_baro_altitude_ft,last_geom_altitude_ft,takeoff_airport_elevation_difference_ft,landing_airport_elevation_difference_ft,pia,ladd,military,interesting,aircraft_type,owner,aircraft_description,category
str,str,str,datetime[ms],str,f64,datetime[ms],str,f64,f64,f64,i64,i64,f64,f64,i64,i64,i64,i64,bool,bool,bool,bool,str,str,str,str
"""a5824e""","""""","""""",2026-03-01 16:09:44.390,"""KE34""",7.463432,2026-03-01 17:26:56.350,"""KFTW""",0.549576,34.850418,-100.902922,20900,21725,32.824821,-97.361569,575,625,18892,-85,false,true,false,false,"""PC12""","""WALKCON HOLDINGS LLC""","""PILATUS PC-12""","""A1"""
"""a5824e""","""""","""""",2026-03-01 22:32:16.070,"""6KS4""",4.089269,2026-03-01 23:35:13.010,"""KX43""",9.657083,38.878707,-101.755554,20850,21375,35.965485,-101.756058,15900,16450,17900,12943,false,true,false,false,"""PC12""","""WALKCON HOLDINGS LLC""","""PILATUS PC-12""","""A1"""


In [9]:
from data_engineering.adsb.read_adsb import read_adsb
from evaluation.flights.visual import visualize_flight


visualize_flight(df_flights.filter(pl.col("icao") == "a5824e"), read_adsb(target_date, icaos=["a5824e"]))